# 03. 집계·병합·정규화

입력: 기존 주택·단속 집계, 인구, 배출량 예측 CSV. 기본 saved 설정은 02번 학습 결과 없이도 실행할 수 있습니다.

새 모델 예측을 쓰려면 02번을 실행한 뒤 PREDICTION_SOURCE를 random_forest 또는 mlp로 바꾸세요.

출력: `merged_raw.csv`, `scaled_rebuilt.csv`, 기존 결과 비교 및 누락 진단. 이 노트북은 모델을 재학습하지 않습니다.

각 노트북은 별도 커널에서 실행할 수 있습니다. 공유 변수는 사용하지 않고 파일로 입력을 읽습니다. ZIP에는 기존 실행 결과도 들어 있습니다.

In [1]:
from pathlib import Path
import sys
# Supports Jupyter launched from the repository root or notebooks/.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'paths.py').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('저장소 폴더 또는 notebooks 폴더에서 노트북을 실행하세요.')
sys.path.insert(0, str(PROJECT_ROOT))
from src.paths import ROOT, data_path, result_path

# 필요 시 주석 해제
# %pip install -r requirements.txt
from IPython.display import Image, HTML, display


In [2]:
"""PDF 기반 재구현. 원본 코드 복원이 아니며 설정/선정 규칙은 새로 명시했다."""
from pathlib import Path
import ast, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from scipy.spatial.distance import cdist
import folium
SEED = 42
PREDICTION_SOURCE = 'saved'
COMMERCIAL_DATASET = 'historical_2022'  # official_2025 선택 가능
COUNTS_SOURCE = 'saved'  # saved: 기존 집계 / rebuilt: 좌표 결과의 재집계
N_CLUSTERS = 3
REMOVE_OUTLIERS = False
OUTLIER_QUANTILE = 0.95
TARGET_DONG = '송천1동'  # PDF의 상세 분석 대상. 새 군집 결과와 별도로 명시

### `read_csv`

In [3]:
def read_csv(name):
    for enc in ('utf-8-sig', 'cp949'):
        try:
            return pd.read_csv(data_path(name), encoding=enc, low_memory=False).dropna(how='all')
        except UnicodeDecodeError:
            continue
    raise ValueError(f'인코딩 확인 필요: {name}')

### `numeric`

In [4]:
def numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(',', '', regex=False), errors='coerce')

### `save_csv`

In [5]:
def save_csv(df, name):
    df.to_csv(result_path(name), index=False, encoding='utf-8-sig')

### `clean_dong`

In [6]:
def clean_dong(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.startswith('['):
        try:
            vals = ast.literal_eval(s)
            s = str(vals[0]) if vals else ''
        except (ValueError, SyntaxError):
            return None
    s = s.split()[-1] if s.split() else ''
    if s.lower() in ('nan', 'none', ''):
        return None
    return s

### `grouped_counts`

In [7]:
def grouped_counts(name, count_name):
    d = read_csv(name)
    d['행정동'] = d['ADDRESS'].map(clean_dong)
    audit = d[d['행정동'].isna()]
    save_csv(audit, name.replace('.csv', '_unmatched.csv'))
    return d.dropna(subset=['행정동']).groupby('행정동').size().rename(count_name).reset_index()

### `build_dataset`

In [8]:
def build_dataset():
    pop = read_csv('전북특별자치도 전주시_인구현황.csv').rename(columns={'구분':'행정동'})
    for c in ['인구수', '세대수']:
        pop[c] = numeric(pop[c])
    pop['행정동'] = pop['행정동'].map(clean_dong)
    predictions = {}
    if PREDICTION_SOURCE != 'saved':
        filename = f'{PREDICTION_SOURCE}_predictions.csv'
        if not result_path(filename).exists():
            raise FileNotFoundError(f'{filename}가 없습니다. 02_model_training.ipynb를 먼저 실행하세요.')
        predictions[PREDICTION_SOURCE] = pd.read_csv(result_path(filename))
    housing = grouped_counts('shit_result.csv', '원룸 및 주택 밀집')
    violations = grouped_counts('holy_result.csv', '쓰레기 민원 빈도')
    # '밀집'은 면적당 밀도가 아니라 원본 발표처럼 건물 건수임.
    # '민원'이라는 원본 컬럼명은 유지하되 실제 데이터는 불법 투기 단속/적발 건수임.
    save_csv(housing, 'housing_counts_rebuilt.csv')
    save_csv(violations, 'violation_counts_rebuilt.csv')
    for generated, filename, old_count, new_count in [
        (housing, '행정동별_빈도수.csv', 'COUNT', '원룸 및 주택 밀집'),
        (violations, '쓰레기 민원_최종.csv', '빈도', '쓰레기 민원 빈도')]:
        old = read_csv(filename).rename(columns={old_count:'saved_count'})
        check = old.merge(generated, on='행정동', how='outer')
        check['difference'] = check[new_count] - check['saved_count']
        save_csv(check, filename.replace('.csv','_comparison.csv'))
    # 원 좌표 파일의 시외 행정동을 별도로 기록. 기존 전처리 규칙은 불명확.
    outside = violations[~violations['행정동'].isin(pop['행정동'])]
    save_csv(outside, 'violation_non_jeonju_dongs.csv')
    if COUNTS_SOURCE == 'saved':
        housing = read_csv('행정동별_빈도수.csv').rename(columns={'COUNT':'원룸 및 주택 밀집'})
        violations = read_csv('쓰레기 민원_최종.csv').rename(columns={'빈도':'쓰레기 민원 빈도'})
    elif COUNTS_SOURCE != 'rebuilt':
        raise ValueError('COUNTS_SOURCE는 saved 또는 rebuilt여야 합니다.')
    if PREDICTION_SOURCE == 'saved':
        saved = read_csv('result.csv').rename(columns={'구분':'행정동'})
        saved = saved.dropna(subset=['행정동','연간 쓰레기 배출량 예측 값'])
        saved['행정동'] = saved['행정동'].map(clean_dong)
        if saved['행정동'].duplicated().any():
            raise ValueError('기존 예측 CSV에 중복 행정동이 있습니다.')
        pred = saved[['행정동','연간 쓰레기 배출량 예측 값']]
    else:
        pred = predictions[PREDICTION_SOURCE][['행정동','연간 쓰레기 배출량 예측 값']]
    raw = pop.merge(housing,on='행정동',how='left',validate='one_to_one').merge(
        violations,on='행정동',how='left',validate='one_to_one').merge(pred,on='행정동',how='left',validate='one_to_one')
    if raw.isna().any().any():
        save_csv(raw[raw.isna().any(axis=1)],'merge_missing.csv')
        raise ValueError('병합 누락 발생. results/tables/merge_missing.csv를 확인하세요.')
    # 실제 인구 비율. 기존 노트북의 max 기반 비율과 구분하여 명명.
    raw['전체 인구 대비 지역 인구 비율'] = raw['인구수'] / raw['인구수'].sum()
    features = ['원룸 및 주택 밀집','쓰레기 민원 빈도','연간 쓰레기 배출량 예측 값']
    scaled = raw.copy()
    scaled[features] = MinMaxScaler().fit_transform(raw[features])
    save_csv(raw,'merged_raw.csv'); save_csv(scaled,'scaled_rebuilt.csv')
    original = read_csv('최종데이터셋_utf8_sig.csv')
    compare = original.merge(scaled[['행정동']+features],on='행정동',suffixes=('_saved','_rebuilt'),validate='one_to_one')
    for c in features: compare[c+'_difference'] = compare[c+'_rebuilt']-compare[c+'_saved']
    save_csv(compare,'original_scaled_comparison.csv')
    return raw, scaled, features

## 입력 선택

In [9]:
PREDICTION_SOURCE = 'saved'  # saved / random_forest / mlp
COUNTS_SOURCE = 'saved'  # saved / rebuilt

## 집계·병합·정규화 실행

In [10]:
raw, scaled, features = build_dataset()
settings = {'prediction_source': PREDICTION_SOURCE, 'counts_source': COUNTS_SOURCE}
result_path('dataset_settings.json').write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding='utf-8')
display(raw.head())
display(scaled[['행정동'] + features].head())

,행정동,인구수,세대수,원룸 및 주택 밀집,쓰레기 민원 빈도,연간 쓰레기 배출량 예측 값,전체 인구 대비 지역 인구 비율
0,중앙동,9646,4885,48,16,6984.578,0.015108
1,풍남동,3549,2242,30,4,6984.578,0.005559
2,노송동,10362,5456,94,22,6984.578,0.016229
3,완산동,4881,2594,44,11,6984.578,0.007645
4,동서학동,5823,3113,37,7,6984.578,0.009120


,행정동,원룸 및 주택 밀집,쓰레기 민원 빈도,연간 쓰레기 배출량 예측 값
0,중앙동,0.023656,0.141176,0.0
1,풍남동,0.010753,0.000000,0.0
2,노송동,0.056631,0.211765,0.0
3,완산동,0.020789,0.082353,0.0
4,동서학동,0.015771,0.035294,0.0
